## PREPARING DATA

In [ ]:
#the import packages
import requests
import pandas as pd
from pandas import json_normalize
import requests
import os
from pathlib import Path
from datetime import datetime, timezone,timedelta,time
from scipy import stats
import json
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
from matplotlib.ticker import MultipleLocator


In [ ]:
pd.set_option("display.max_columns", None)

In [ ]:
def loadDataFromFile(file_name):
    script_dir = Path().resolve().parent
    print(script_dir)
    data_folder = script_dir /'data'
    print(data_folder)
    data_folder.mkdir(exist_ok=True)
    
    file_path = data_folder / (file_name + ".json")
    
    if file_path.exists():
        df = pd.read_json(file_path)
        print(f"Loaded {len(df)} records from {file_path}")
        return df
    else:
        print(f"File {file_path} does not exist.")
        return None    

In [ ]:
userInputDataRaw = loadDataFromFile("UserPrevious experiments-preprocessed")
timeSeriesData_BIGraw = loadDataFromFile("Data:Previous experiments-preprocessed")

In [ ]:
timeSeriesData_BIGraw = timeSeriesData_BIGraw.set_index("seconds",drop=False)

In [ ]:
a = userInputDataRaw.index
b = timeSeriesData_BIGraw["keys"].unique()
diff_all = list(set(a).symmetric_difference(set(b)))
print(diff_all)  
userInputDataRaw.index = timeSeriesData_BIGraw["keys"].unique()
print(userInputDataRaw.index)

In [ ]:
# Convert back to timedelta
timeSeriesData_BIGraw['timestamp'] = pd.to_timedelta(timeSeriesData_BIGraw['timestamp'], unit='ms')

# Convert back to datetime

timeSeriesData_BIGraw ["datetime_timestamp"]= timeSeriesData_BIGraw['datetime_timestamp'].transform(
    lambda x: pd.to_datetime(x, unit='ms')
)


columns_datetime= [
       'date of experiment', 'actual timestamp StartingExperiment',
       'actual timestamp EndingExperiment']
columns_timedelta = ['time taken total','time taken before insertion',
       'timestamp InsertingSource timedelta',
       'time taken after insertion']
# Ensure target columns are of object type before assignment
userInputDataRaw[columns_datetime] = userInputDataRaw[columns_datetime].astype('object')
userInputDataRaw[columns_timedelta] = userInputDataRaw[columns_timedelta].astype('object')

userInputDataRaw.loc[:,columns_datetime] = userInputDataRaw.loc[:,columns_datetime].apply(lambda x:pd.to_datetime(x, unit='ms'))
userInputDataRaw.loc[:,columns_timedelta] = userInputDataRaw.loc[:,columns_timedelta].apply(lambda x:pd.to_timedelta(x, unit='ms'))

In [ ]:
timeSeriesData_BIG = timeSeriesData_BIGraw.copy()
userInputData = userInputDataRaw.copy()

In [ ]:
userInputData

In [ ]:
userInputData["room"].unique()

In [ ]:
#keep the data from the last set experiments made that have the 3 sensors in a triangle shape,they have 16 particular points in the space d
room_mask = userInputData["room"].isin(['Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0.5  ,id:2 Π:0.4 Α:1.0'])
#open_door_mask = userInputData["are-doors-opened"] != "on"

mask = room_mask # & open_door_mask
userInputData = userInputData.loc[mask]
#grab all the data that are contained in those experiments
timeSeriesData_BIG = timeSeriesData_BIG.loc[timeSeriesData_BIG["keys"].isin(userInputData.index)]

In [ ]:
userInputData

In [ ]:
timeSeriesData_BIG

In [ ]:
column_to_transform = "x-y axis"
userInputData.loc[:,column_to_transform] = userInputData.loc[:,column_to_transform].apply(tuple)

In [ ]:
userInputData["x-y axis"].unique()

In [ ]:
#from the experiments with door open, drop the positions which doesn't fit to the 16 source position we are going to check the model
#also drop the experiments with the None values
axis_list = [(None, None), (-2.15, 2.0), (-1.35, 2.0), (-1.35, 1.5),
       (-1.35, 1.0), (-1.85, 1.0), (-1.85, 1.5)]
axis_mask = ~userInputData["x-y axis"].isin(axis_list)

mask = axis_mask 
userInputData = userInputData.loc[mask]
#grab all the data that are contained in those experiments
timeSeriesData_BIG = timeSeriesData_BIG.loc[timeSeriesData_BIG["keys"].isin(userInputData.index)]

In [ ]:
userInputData

In [ ]:
timeSeriesData_BIG

In [ ]:
userInputData.loc[userInputData["are-doors-opened"]=="on"].shape

In [ ]:
userInputData.loc[userInputData["are-doors-opened"]!="on"].shape

In [ ]:
max_before= -30
max_after = 299

column_to_check = "time taken before insertion (seconds)-capped"
mask = userInputData[column_to_check] >max_before
print(f"{column_to_check} < {max_before}:\n {userInputData.loc[mask,column_to_check]}")

column_to_check = "time taken after insertion (seconds)-capped"
mask = userInputData[column_to_check] < max_after
print(f"{column_to_check} < {max_after}:\n {userInputData.loc[mask,column_to_check]}")
print(f"userInputData rows {userInputData.shape[0]}")

In [ ]:
max_before= -30
max_after = 299

mask_before = (userInputData["time taken before insertion (seconds)-capped"] == max_before)
max_after = (userInputData["time taken after insertion (seconds)-capped"] == max_after)
mask = mask_before & max_after

userInputData = userInputData.loc[mask,:].copy()
print(f"userInputData rows {userInputData.shape[0]}")
timeSeriesData_BIG = timeSeriesData_BIG.loc[timeSeriesData_BIG["keys"].isin(userInputData.index)]

## PRINT DATA PER SENSOR

### Split back into dict
dict_of_timeseries = {k: v.drop(columns="keys").reset_index(drop=True) 
             for k, v in timeSeriesData_BIG.groupby("keys")}
for index,data in dict_of_timeseries.items():
    dict_of_timeseries[index] = dict_of_timeseries[index].set_index("seconds",drop=False)

In [ ]:
def plot_position(userInputData,sample_row_of_the_group):
    plt.figure(figsize=(6, 6))  
    position_of_sensors = userInputData.iloc[-1]
    all_positions = userInputData.loc[:, ["x axis", "y axis"]]
    # Extra points
    extra_positions = np.array([
        [position_of_sensors["position of Id=0:BME680:breathVocEquivalent-x axis"], position_of_sensors["position of Id=0:BME680:breathVocEquivalent-y axis"]],
    
        [position_of_sensors["position of Id=1:BME680:breathVocEquivalent-x axis"], position_of_sensors["position of Id=1:BME680:breathVocEquivalent-y axis"]],
        [position_of_sensors["position of Id=2:BME680:breathVocEquivalent-x axis"], position_of_sensors["position of Id=2:BME680:breathVocEquivalent-y axis"]]
    ])
    extra_ids = ["id0","id1", "id2"]
    
    
    room_length = 4.0
    room_width = 3.25
    
    
    # Create scatterplot of the sources of all the particular setup
    #sns.scatterplot(x=positions[:,0], y=positions[:,1])
    sns.scatterplot(data=all_positions, x="x axis", y="y axis", color='blue', s=100, label='User Input Data')
    
    
    
    # Add the positions of sensors
    sns.scatterplot(x=extra_positions[:,0], y=extra_positions[:,1], color='red', s=100)
    x_sensor_highlight = sample_row_of_the_group["x axis"]
    y_sensor_highlight = sample_row_of_the_group["y axis"]
    # Plot a hollow circle around it
    plt.scatter(x_sensor_highlight, y_sensor_highlight, s=500, facecolors='none', edgecolors='green', linewidths=2, label='Highlighted point')  
    # Draw lines and annotate distances
    distances_from_sensors = (
        sample_row_of_the_group["Euclidian distance to Id=0:BME680:breathVocEquivalent"],
        sample_row_of_the_group["Euclidian distance to Id=1:BME680:breathVocEquivalent"],
        sample_row_of_the_group["Euclidian distance to Id=2:BME680:breathVocEquivalent"]
    )
    
    for i, (x, y) in enumerate(extra_positions):
        plt.plot([x_sensor_highlight, x], [y_sensor_highlight, y], color='red', linewidth=0.7, alpha=0.7)
        
        if distances_from_sensors is not None:
            # Midpoint of the line for annotation
            mid_x = (x_sensor_highlight + x) / 2
            mid_y = (y_sensor_highlight + y) / 2
            plt.text(mid_x, mid_y, f"{distances_from_sensors[i]:.2f}", color='red', fontsize=8, ha='center', va='center',
                         bbox=dict(facecolor='white', edgecolor='none', alpha=0.6, pad=1))
    # Annotate extra points with their IDs
    for i, (x, y) in enumerate(extra_positions):
        plt.text(x, y, extra_ids[i], fontsize=12, ha='right', va='bottom', color='red')
    
    
    # Set axis limits
    plt.xlim(-room_width, 0)
    plt.ylim(0, room_length)
    
    # Add grid
    plt.grid(True, which="both", linestyle="--", linewidth=0.7, alpha=0.7)
    # Smaller legend
    plt.legend(fontsize=8, markerscale=0.8, labelspacing=0.4)

    plt.show()

In [ ]:
def printDataBasedOnDate(column_to_print,userInputData,timeSeriesData_BIG,room_other_grouping,type_of_other_grouping):
    
    column_names_keys_color_values = {"Id=0:BME680:breathVocEquivalent":"blue","Id=1:BME680:breathVocEquivalent":"green","Id=2:BME680:breathVocEquivalent":"yellow"}
    
    for group_name,indexes_of_the_group in room_other_grouping.items(): 
        timeSeriesData_BIG_copy = timeSeriesData_BIG.copy() 

        if ("position"  in type_of_other_grouping):
            sample_row_of_the_group = userInputData.loc[indexes_of_the_group[0],:]
            
            plot_position(userInputData,sample_row_of_the_group)       
        print(f"group_name {group_name}")
        print(f"indexes_of_the_group {indexes_of_the_group}")
        data = timeSeriesData_BIG_copy.loc[timeSeriesData_BIG_copy["keys"].isin(indexes_of_the_group),:]  
      # Create relplot
        g = sns.relplot(
            data=data,
            x="seconds",
            y=column_to_print,
            hue="sensors",
            col="keys",        # separate subplot per key
            kind="line",
            col_wrap=3, 
                height=7,    # default = 5
            aspect=1, # width = height × aspect (so 6 × 1.5 = 9 inches wide per subplot
            palette=column_names_keys_color_values,  # <<< ensures the same colors across all subplots  
            linewidth=2,
           facet_kws={
            "sharex": False,
            "sharey": False       
    
        })
        

        # >>> ADD THIS PART <<<
        for ax in g.axes.flat:
            ax.xaxis.set_major_locator(MultipleLocator(30))
            ax.xaxis.set_minor_locator(MultipleLocator(10))
            ax.grid(True, which='both', linestyle=':', linewidth=0.5)
            
 
        
    # Get the horizontal and  vertical line position for this experiment
        for key_value, ax in g.axes_dict.items():
           
                #value to show the time that source is inserted
          
            userInputDataRow = userInputData.loc[key_value,:]
        #    x_position = f"side-right-wall {userInputDataRow['side-right-wall']},side-left-wall {userInputDataRow['side-left-wall']} \n"
        #    y_position = f"front-wall {userInputDataRow['front-wall']},back-wall {userInputDataRow['back-wall']} \n"
            
            euclidian_distances = (
                                  f"distance from Id0 sensor {userInputDataRow['Euclidian distance to Id=0:BME680:breathVocEquivalent']}\n",
                                  f"distance from Id1 sensor {userInputDataRow['Euclidian distance to Id=1:BME680:breathVocEquivalent']}\n",
                                  f"distance from Id2 sensor {userInputDataRow['Euclidian distance to Id=2:BME680:breathVocEquivalent']}\n",
            )
            subtitle=  (
                        f"At experiment with key {key_value}\n datetime:{userInputDataRow['actual timestamp StartingExperiment']}\n", 
                        f"experimentState:{userInputDataRow['experimentState']}\n",
                        f"x-axis: {userInputDataRow['x axis']} , y-axis: {userInputDataRow['y axis']}\n"
            )
            if ("distance"  in type_of_other_grouping):
               subtitle = subtitle + "\n"+euclidian_distances  
            ax.set_title(subtitle, fontsize=9)   
            g.fig.suptitle(f"Group: {group_name}", fontsize=16)
        
            g.fig.subplots_adjust(
                    top=0.75,   # space for overall title
                    wspace=0.2, # horizontal space between subplots
                    hspace=0.3 # vertical space between subplots
                   )

        plt.show()   
             

In [ ]:
def plot_all_positions(userInputData):
    room_other_grouping = userInputData.groupby(["x axis","y axis"]).groups
    
    for group_name,indexes_of_the_group in room_other_grouping.items(): 
        sample_row_of_the_group = userInputData.loc[indexes_of_the_group[0],:]
        plot_position(userInputData,sample_row_of_the_group)      
        
plot_all_positions(userInputData)

sensors = timeSeriesData_BIG["sensors"].unique()
euclidian_distances_columns = [f"Euclidian distance to {sensor}" for sensor in sensors ]
group_by_list = ["room","experimentState",*euclidian_distances_columns]
room_other_grouping = userInputData.groupby(group_by_list).groups
type_of_other_grouping = ["experimentState","position","distance"]

sensors = timeSeriesData_BIG["sensors"].unique()
for sensor in sensors:
    mask = timeSeriesData_BIG["sensors"] == sensor
    timeSeriesData_BIG_subset = timeSeriesData_BIG.loc[mask,:]
    print(sensor)
    printDataBasedOnDate("VOC",userInputData,timeSeriesData_BIG_subset,room_other_grouping,type_of_other_grouping)

sensors = timeSeriesData_BIG["sensors"].unique()
euclidian_distances_columns = [f"Euclidian distance to {sensor}" for sensor in sensors ]
group_by_list = ["room","experimentState",*euclidian_distances_columns]
room_other_grouping = userInputData.groupby(group_by_list).groups
type_of_other_grouping = ["experimentState","position","distance"]

sensors = timeSeriesData_BIG["sensors"].unique()
for sensor in sensors:
    mask = timeSeriesData_BIG["sensors"] == sensor
    timeSeriesData_BIG_subset = timeSeriesData_BIG.loc[mask,:]
    print(sensor)
    printDataBasedOnDate("VOC rolling average",userInputData,timeSeriesData_BIG_subset,room_other_grouping,type_of_other_grouping)

sensors = timeSeriesData_BIG["sensors"].unique()
euclidian_distances_columns = [f"Euclidian distance to {sensor}" for sensor in sensors ]
group_by_list = ["room","experimentState"]
room_other_grouping = userInputData.groupby(group_by_list).groups
type_of_other_grouping = ["experimentState","date of experiment"]

sensors = timeSeriesData_BIG["sensors"].unique()
for sensor in sensors:
    mask = timeSeriesData_BIG["sensors"] == sensor
    timeSeriesData_BIG_subset = timeSeriesData_BIG.loc[mask,:]
    print(sensor)
    printDataBasedOnDate("VOC rolling average",userInputData,timeSeriesData_BIG_subset,room_other_grouping,type_of_other_grouping)

## MODEL TRAINING

### CREATE THE X AND Y ARRAYS

In [ ]:

df_filtered = timeSeriesData_BIG.loc[timeSeriesData_BIG["keys"].isin(userInputData.index)]
dfs_by_sensor = {
    sensor: g
    for sensor, g in df_filtered.groupby("sensors")
}


In [ ]:
dfs_by_sensor

In [ ]:
dfs_by_sensor['Id=0:BME680:breathVocEquivalent'].columns

In [ ]:
sensors = timeSeriesData_BIG["sensors"].unique()

euclidian_distances_columns = [f"Euclidian distance to {sensor}" for sensor in sensors ]
userInputData.loc[:,euclidian_distances_columns] = userInputData.loc[:,euclidian_distances_columns].round(2)

In [ ]:
print(userInputData["x-y axis"].unique())

In [ ]:
columns_to_keep = ["VOC rolling average"]

key_to_grab_size = userInputData.index[0]
mask = dfs_by_sensor['Id=0:BME680:breathVocEquivalent']["keys"] == key_to_grab_size

sample_data = dfs_by_sensor['Id=0:BME680:breathVocEquivalent'].loc[mask,columns_to_keep]
print(sample_data.shape[0])


In [ ]:


columns_to_create_X_data = "VOC rolling average"

column_to_create_Y_data = "x-y axis"

sensors = timeSeriesData_BIG["sensors"].unique()
X ={}
Y ={}

#grab the number of features

key_to_grab_size = userInputData.index[0]
mask = dfs_by_sensor['Id=0:BME680:breathVocEquivalent']["keys"] == key_to_grab_size

sample_data = dfs_by_sensor['Id=0:BME680:breathVocEquivalent'].loc[mask,columns_to_create_X_data]
X_subset_columns_size = sample_data.shape[0]

dict_flattened_arrays_per_sensor_per_distance ={}
for sensor in sensors:
    X[sensor] = np.empty((0, X_subset_columns_size))
    Y[sensor] = np.empty((0, 2))
    dict_flattened_arrays_per_sensor_per_distance[sensor] = {}
    
    for X_Y_axis,experiments in userInputData.groupby(column_to_create_Y_data):
        
        rows_size = experiments.shape[0]
        Y_distance_subset =np.empty((rows_size,2))
        X_distance_subset = np.empty((rows_size,X_subset_columns_size))

        
        Y_distance_subset[:,0] = X_Y_axis[0]
        Y_distance_subset[:,1] = X_Y_axis[1]    
        Y[sensor] = np.vstack((Y[sensor],Y_distance_subset)) 
        
             
        for array_index,experiment_index in enumerate(experiments.index):
          
            mask = dfs_by_sensor[sensor]["keys"]== experiment_index
            flatten_array = dfs_by_sensor[sensor].loc[mask,columns_to_create_X_data].to_numpy().reshape(1, -1)
            
            X_distance_subset[array_index,:] =   flatten_array
            
        X[sensor] = np.vstack((X[sensor],X_distance_subset))   
            



In [ ]:
X

In [ ]:
Y

In [ ]:
X.keys()

### DISCOVER POTENTIAL DIMENSIONALITY REDUCTION

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

In [ ]:


sensor_names = ['Id=0:BME680:breathVocEquivalent','Id=1:BME680:breathVocEquivalent','Id=2:BME680:breathVocEquivalent']


for sensor_name in sensor_names:
    print(sensor_name)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X[sensor_name])
    print(scaler.get_params())
    # Step 2: Apply PCA
    pca = PCA()
    X_pca = pca.fit_transform(X_scaled)
    
    # Step 3: Explained variance
    explained_variance = pca.explained_variance_ratio_
    cumulative_variance = np.cumsum(explained_variance)
    
    # Only display first 10 components max
    max_components = min(10, len(explained_variance))
    ev_to_plot = explained_variance[:max_components]
    cum_to_plot = cumulative_variance[:max_components]
    
    # Step 4: Bar plot of cumulative explained variance
    plt.figure(figsize=(8,5))
    plt.bar(range(1, max_components + 1), cum_to_plot)
    plt.xlabel('Number of PCA components')
    plt.ylabel('Cumulative explained variance')
    plt.title('Cumulative explained variance (first 10 components)')
    plt.grid(True, axis='y')
    plt.show()
    
    # Step 5: Optimal number of components for ~90% variance
    optimal_components = np.argmax(cumulative_variance >= 0.90) + 1
    print("Optimal number of components to explain ~90% variance:", optimal_components)


### SEARCH BEST MODEL OF EACH INDIVIDUAL SENSOR'S X AND Y AXIS

In [ ]:
# Required imports
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error,mean_absolute_error, r2_score


In [ ]:

# 1. Linear Regression (no hyperparameters to tune, just for completeness)
lr = LinearRegression()
lr_params = {}  # No parameters for simple LinearRegression

# 2. Ridge Regression
ridge = Ridge()
ridge_params = {
    'alpha': [0.01, 0.1, 1, 10, 100],
    'solver': ['auto', 'svd', 'cholesky', 'lsqr']
}

# 3. Lasso Regression
lasso = Lasso()
lasso_params = {
    'max_iter':[10000],
    'alpha': [0.001, 0.01, 0.1, 1, 10],
}

# 4. ElasticNet
elastic = ElasticNet()
elastic_params = {
    'max_iter':[10000],
    'alpha': [0.001, 0.01, 0.1, 1, 10],
    'l1_ratio': [0.1, 0.5, 0.7, 0.9]
}

# 5. Support Vector Regression
svr = SVR()
svr_params = {
    'kernel':["rbf"],
    "C": [0.1, 1, 10, 100],
    "epsilon": [0.001, 0.01, 0.1, 0.2],
    "gamma": ["scale", "auto"]
}


# 7. Random Forest Regression
rf = RandomForestRegressor()
rf_params = {
    'random_state':[42],
    'n_estimators': [10, 50, 100,200],
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# 8. Gradient Boosting Regression
gbr = GradientBoostingRegressor()
gbr_params = {
    'random_state':[42],
    'n_estimators': [10, 50, 100,200],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [ 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

models = {
    'LinearRegression': (lr, lr_params),
    'Ridge': (ridge, ridge_params),
    'Lasso': (lasso, lasso_params),
    'ElasticNet': (elastic, elastic_params),
    'SVR': (svr, svr_params),
    #'DecisionTree': (dt, dt_params),
  #  'RandomForest': (rf, rf_params),
    #'GradientBoosting': (gbr, gbr_params)
}

PCA__n_components = [3,4,5,6,8,10]


In [ ]:

def run_grid_search_per_model(X_train, y_train,cv_number,verbose,name,model,params,PCA__n_components):
        estimators_with_no_scaling_need = ['RandomForest','DecisionTree','GradientBoosting']

        results = {}
        if name in estimators_with_no_scaling_need:
             pipe = Pipeline([
                 ('model', model)
             ]) 
        else:
             pipe = Pipeline([
                 ('scaler', StandardScaler()),
                 ('PCA', PCA()),
                 ('model', model)
             ])
        # Build a correct param grid:
        param_grid = {
            **{'model__' + k: v for k, v in params.items()}
        }
        if name not in estimators_with_no_scaling_need:
            param_grid['PCA__n_components'] = PCA__n_components
            
        grid = GridSearchCV(
            estimator=pipe,
            param_grid=param_grid,
            cv=cv_number,
            n_jobs=-1,
            scoring='neg_mean_squared_error',
            verbose=2
        )
     
        grid.fit(X_train, y_train)
    
        results["name"]  = name
        results["model"] = grid.best_estimator_
        results["parameters"] = grid.best_params_
        results["score"] = grid.best_score_
    
        return results

def run_grid_search_find_optimal_model_per_sensor(X_train, y_train,cv_number,verbose,models,PCA__n_components):
    #put a score way worse than abest_score_nything we should have
    best_result = {
        "score" : -10,
        "model" : '',
        "parameters" : {}    
    }
    for name, (model, params) in models.items():
        print(f"Running GridSearchCV for {name}...")

        results = run_grid_search_per_model(X_train, y_train,cv_number,verbose,name,model,params,PCA__n_components)
        print(results)

        if (results["score"] >  best_result["score"]) :
            best_result["name"] = results["name"]
            print(best_result["name"])
            best_result["score"] = results["score"]
            print(best_result["score"])
            best_result["model"] = results["model"]
            print(best_result["model"])
            best_result["parameters"] = results["parameters"]
            print(best_result["parameters"])
    return best_result

In [ ]:
sensor_names = [
    "Id=0:BME680:breathVocEquivalent",
    "Id=1:BME680:breathVocEquivalent",
    "Id=2:BME680:breathVocEquivalent"    
]

test_size = 0.2         # number of data taken
random_state = 42         # predefined random state for reproducibility
cv_number = 5
verbose =2
X_AXIS_best_result_per_sensor = {}
results_of_grid_search = {}
X_AXIS_best_result_per_sensor = {}
for sensor_name in sensor_names:
    X_arr = np.asarray(X[sensor_name])
    Y_arr_full = np.asarray(Y[sensor_name])
    Y_arr_Current_Axis = 0
    Y_arr = Y_arr_full[:,Y_arr_Current_Axis]

    # --- Train/test split ---
    X_train, X_test, y_train, y_test = train_test_split(
        X_arr, Y_arr, test_size=test_size, random_state=random_state, shuffle=True
    )
    X_AXIS_best_result_per_sensor[sensor_name] = run_grid_search_find_optimal_model_per_sensor(X_train,y_train,cv_number,verbose,models,PCA__n_components)

In [ ]:
X_AXIS_best_result_per_sensor

In [ ]:
sensor_names = [
    "Id=0:BME680:breathVocEquivalent",
    "Id=1:BME680:breathVocEquivalent",
    "Id=2:BME680:breathVocEquivalent"    
]

test_size = 0.2         # number of data taken
random_state = 42         # predefined random state for reproducibility
cv_number = 5
verbose =2

Y_AXIS_best_result_per_sensor = {}

for sensor_name in sensor_names:
    X_arr = np.asarray(X[sensor_name])
    Y_arr_full = np.asarray(Y[sensor_name])
    Y_arr_Current_Axis = 1
    Y_arr = Y_arr_full[:,Y_arr_Current_Axis]
    print(f"running grid search for sensor:{sensor_name}")
    # --- Train/test split ---
    
    X_train, X_test, y_train, y_test = train_test_split(
        X_arr, Y_arr, test_size=test_size, random_state=random_state, shuffle=True
    )
    Y_AXIS_best_result_per_sensor[sensor_name] = run_grid_search_find_optimal_model_per_sensor(X_train,y_train,cv_number,verbose,models,PCA__n_components)

In [ ]:
Y_AXIS_best_result_per_sensor

### IMPLEMENT THE BEST FOUND TUNED MODELS

In [ ]:
sensor_names = [
    "Id=0:BME680:breathVocEquivalent",
    "Id=1:BME680:breathVocEquivalent",
    "Id=2:BME680:breathVocEquivalent"    
]
X_AXIS_X_TRAIN = {}
X_AXIS_X_TEST = {}
X_AXIS_Y_TRAIN = {}
X_AXIS_Y_TEST = {}

Y_AXIS_X_TRAIN = {}
Y_AXIS_X_TEST = {}
Y_AXIS_Y_TRAIN = {}
Y_AXIS_Y_TEST = {}
for sensor_name in sensor_names:

    X_arr = np.asarray(X[sensor_name])
    y_arr = np.asarray(Y[sensor_name])

    # --- Train/test split  X AXIS---
    X_AXIS_X_TRAIN[sensor_name], X_AXIS_X_TEST[sensor_name], X_AXIS_Y_TRAIN[sensor_name], X_AXIS_Y_TEST[sensor_name] = train_test_split(
        X_arr, y_arr[:,0], test_size=test_size, random_state=random_state, shuffle=True
    )

      # --- Train/test split  X AXIS---
    Y_AXIS_X_TRAIN[sensor_name], Y_AXIS_X_TEST[sensor_name], Y_AXIS_Y_TRAIN[sensor_name], Y_AXIS_Y_TEST[sensor_name] = train_test_split(
        X_arr, y_arr[:,1], test_size=test_size, random_state=random_state, shuffle=True
    )
    scaler_x_axis = StandardScaler()
    X_AXIS_X_TRAIN[sensor_name] = scaler_x_axis.fit_transform(X_AXIS_X_TRAIN[sensor_name]) 
    X_AXIS_X_TEST[sensor_name]  = scaler_x_axis.transform(X_AXIS_X_TEST[sensor_name])

    scaler_y_axis = StandardScaler()
    Y_AXIS_X_TRAIN[sensor_name] = scaler_x_axis.fit_transform(Y_AXIS_X_TRAIN[sensor_name]) 
    Y_AXIS_X_TEST[sensor_name]  = scaler_x_axis.transform(Y_AXIS_X_TEST[sensor_name]) 

    pca_parameters_x_axis = X_AXIS_best_result_per_sensor[sensor_name]['parameters']["PCA__n_components"]
    pca_x_axis = PCA(n_components=pca_parameters_x_axis)
    print(f"{sensor_name}:{pca_x_axis}")

    X_AXIS_X_TRAIN[sensor_name] = pca_x_axis.fit_transform(X_AXIS_X_TRAIN[sensor_name])
    X_AXIS_X_TEST[sensor_name]  = pca_x_axis.transform(X_AXIS_X_TEST[sensor_name])

    pca_parameters_y_axis = Y_AXIS_best_result_per_sensor[sensor_name]['parameters']["PCA__n_components"]

    pca_y_axis = PCA(n_components=pca_parameters_y_axis)
    print(f"{sensor_name}:{pca_y_axis}")

    Y_AXIS_X_TRAIN[sensor_name] = pca_y_axis.fit_transform(Y_AXIS_X_TRAIN[sensor_name])
    Y_AXIS_X_TEST[sensor_name]  = pca_y_axis.transform(Y_AXIS_X_TEST[sensor_name])

In [ ]:
X_AXIS_X_TRAIN

In [ ]:
X_AXIS_X_TEST

In [ ]:
TRAINED_MODELS_X_AXIS= {}
TRAINED_MODELS_Y_AXIS ={}
for sensor_name in sensor_names:
    #FIT FOR X_AXIS
    dict_to_get_model_and_params = X_AXIS_best_result_per_sensor
    model = dict_to_get_model_and_params[sensor_name]['model']
    params = {
    k.replace("model__", ""): v
    for k, v in dict_to_get_model_and_params[sensor_name]['parameters'].items()
    if k.startswith("model")
    }
    TRAINED_MODELS_X_AXIS[sensor_name] = model.fit(X_AXIS_X_TRAIN[sensor_name],X_AXIS_Y_TRAIN[sensor_name])
    
    #FIT FOR Y_AXIS
    dict_to_get_model_and_params = Y_AXIS_best_result_per_sensor
    model = dict_to_get_model_and_params[sensor_name]['model']
    params = {
        k.replace("model__", ""): v
        for k, v in dict_to_get_model_and_params[sensor_name]['parameters'].items()
        if k.startswith("model")
    }
    #TRAINED_MODELS_Y_AXIS[sensor_name] = model.set_params(**params)
    TRAINED_MODELS_Y_AXIS[sensor_name] = model.fit(Y_AXIS_X_TRAIN[sensor_name],Y_AXIS_Y_TRAIN[sensor_name])


In [ ]:
TRAINED_MODELS_X_AXIS['Id=0:BME680:breathVocEquivalent']

### SEE RESULTS

In [ ]:
def plot_pred_vs_actual(y_test, y_pred,sensor_name,AXIS):
    # --- Plot: predicted vs actual ---
    mse = mean_squared_error(y_test, y_pred)
    
    r2  = r2_score(y_test, y_pred)
    #check if a numpy array is 1D,if yes,make it 2D with one n,1
    if y_test.ndim == 1:
       y_test =  y_test.reshape(-1,1)
    if y_pred.ndim == 1:
       y_pred =  y_pred.reshape(-1,1)   

    EucDis = np.linalg.norm(y_test - y_pred,axis=1).mean()
    print(f"MSE on test set: {mse:.4f}")
    
    print(f"R^2 on test set: {r2:.4f}")

    print(f"Euclidian distance median value {EucDis:.4f}")
  

    if (y_test.shape[1] == 1) and (y_pred.shape[1] == 1):
        plt.figure(figsize=(7,6))
        plt.scatter(y_test, y_pred, s=50)
        plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], linestyle='--')
        plt.xlabel("True target")
        plt.ylabel("Predicted target")
        plt.title(f"{sensor_name} {AXIS} predictions vs actual")
        plt.grid(True)
        plt.show()

    else:
        # --- Plot ---
        plt.scatter(y_test[:,0], y_test[:,1], label="True", alpha=0.6)
        plt.scatter(y_pred[:,0], y_pred[:,1], label="Predicted", alpha=0.6)
        plt.legend()
        plt.xlabel("X")
        plt.ylabel("Y")
        plt.title("True vs Predicted Positions")
        plt.grid(True)
        plt.show()


In [ ]:
#PREDICT AND SEE INDIVIDUAL RESULTS
PREDICT_X_AXIS = {}
PREDICT_Y_AXIS = {}
PREDICT_POSITION_VALUE = {}
for sensor_name in sensor_names:
    #predict x axis
    PREDICT_X_AXIS[sensor_name] = TRAINED_MODELS_X_AXIS[sensor_name].predict(X_AXIS_X_TEST[sensor_name])

    plot_pred_vs_actual(X_AXIS_Y_TEST[sensor_name],PREDICT_X_AXIS[sensor_name],sensor_name,"X_AXIS")

    #predict y axis

    PREDICT_Y_AXIS[sensor_name] = TRAINED_MODELS_Y_AXIS[sensor_name].predict(Y_AXIS_X_TEST[sensor_name])

    plot_pred_vs_actual(Y_AXIS_Y_TEST[sensor_name],PREDICT_Y_AXIS[sensor_name],sensor_name,"Y_AXIS")
    PREDICT_POSITION_VALUE[sensor_name] =  np.column_stack(( PREDICT_X_AXIS[sensor_name], PREDICT_Y_AXIS[sensor_name]))


GRAB THE MEDIAN OF EACH REGRESSOR

In [ ]:
# PREDICTION_VALUES_MEDIAN = {
#     "MSE": '',
#     "R2" : '',
#     "Median Euclidian Distance":''

# }
sensor_names = [
    "Id=0:BME680:breathVocEquivalent",
    "Id=1:BME680:breathVocEquivalent",
    "Id=2:BME680:breathVocEquivalent"    
]
#the random state is the same, so we know they have they same amount taken from y test and afterwars, they have the same amount of elements.
POSITION_Y_TRAIN = np.column_stack((X_AXIS_Y_TRAIN[sensor_names[0]],Y_AXIS_Y_TRAIN[sensor_names[0]]))
POSITION_Y_TEST  = np.column_stack((X_AXIS_Y_TEST[sensor_names[0]],Y_AXIS_Y_TEST[sensor_names[0]]))
row_number = POSITION_Y_TEST.shape[0]
PREDICT_POSITION_VALUE = np.empty((row_number, 2)) 

PREDICT_POSITION_VALUE[:,0] = np.column_stack(([column for column in PREDICT_X_AXIS.values()])).mean(axis=1)

PREDICT_POSITION_VALUE[:,1] = np.column_stack(([column for column in PREDICT_Y_AXIS.values()])).mean(axis=1)

plot_pred_vs_actual(POSITION_Y_TEST, PREDICT_POSITION_VALUE,sensor_names,"X-Y AXIS")

### TRAIN A STACKED REGRESSION MODEL 

In [ ]:
from sklearn.ensemble import StackingRegressor

In [ ]:

# 1. Linear Regression (no hyperparameters to tune, just for completeness)
lr = LinearRegression()
lr_params = {}  # No parameters for simple LinearRegression

# 2. Ridge Regression
ridge = Ridge()
ridge_params = {
    'alpha': [0.01, 0.1, 1, 10, 100],
    'solver': ['auto', 'svd', 'cholesky', 'lsqr']
}

# 3. Lasso Regression
lasso = Lasso()
lasso_params = {
    'max_iter':[10000],
    'alpha': [0.001, 0.01, 0.1, 1, 10],
}

# 4. ElasticNet
elastic = ElasticNet()
elastic_params = {
    'max_iter':[10000],
    'alpha': [0.001, 0.01, 0.1, 1, 10],
    'l1_ratio': [0.1, 0.5, 0.7, 0.9]
}

# 5. Support Vector Regression
svr = SVR()
svr_params = {
    'kernel':["rbf"],
    "C": [0.1, 1, 10, 100],
    "epsilon": [0.001, 0.01, 0.1, 0.2],
    "gamma": ["scale", "auto"]
}


# 7. Random Forest Regression
rf = RandomForestRegressor()
rf_params = {
    'random_state':[42],
    'n_estimators': [10, 50, 100,200],
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# 8. Gradient Boosting Regression
gbr = GradientBoostingRegressor()
gbr_params = {
    'random_state':[42],
    'n_estimators': [10, 50, 100,200],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [ 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

models = {
    #'LinearRegression': (lr, lr_params),
    #'Ridge': (ridge, ridge_params),
    'Lasso': (lasso, lasso_params),
    'ElasticNet': (elastic, elastic_params),
    'SVR': (svr, svr_params),
    #'DecisionTree': (dt, dt_params),
    #'RandomForest': (rf, rf_params),
    #'GradientBoosting': (gbr, gbr_params)
}

PCA__n_components = [3,4,6,8,10]


In [ ]:

#Find the best parameters per model per sensor
sensor_names = [
    "Id=0:BME680:breathVocEquivalent",
    "Id=1:BME680:breathVocEquivalent",
    "Id=2:BME680:breathVocEquivalent"    
]
verbose = 1
test_size=0.2
cv_number = 5
random_state = 42
best_models_X_AXIS={}
for sensor_name in sensor_names:
    
    X_arr = np.asarray(X[sensor_name])
    Y_arr_full = np.asarray(Y[sensor_name])
    Y_arr_Current_Axis = 0
    Y_arr = Y_arr_full[:,Y_arr_Current_Axis]
    print(f"running grid search for sensor:{sensor_name}")
    # --- Train/test split ---
    X_train, X_test, y_train, y_test = train_test_split(
        X_arr, Y_arr, test_size=test_size, random_state=random_state, shuffle=True
    )
    best_models_X_AXIS[sensor_name] = {} 
    for name, (model, params) in models.items():
            results = run_grid_search_per_model(X_train, y_train,cv_number,verbose,name,model,params,PCA__n_components)
            best_models_X_AXIS[sensor_name][results["name"]] = results


In [ ]:
best_models_X_AXIS

In [ ]:

#Find the best parameters per model per sensor
sensor_names = [
    "Id=0:BME680:breathVocEquivalent",
    "Id=1:BME680:breathVocEquivalent",
    "Id=2:BME680:breathVocEquivalent"    
]
verbose = 1
cv_number = 5
random_state = 42
best_models_Y_AXIS={}
for sensor_name in sensor_names:
    
    X_arr = np.asarray(X[sensor_name])
    Y_arr_full = np.asarray(Y[sensor_name])
    Y_arr_Current_Axis = 1
    Y_arr = Y_arr_full[:,Y_arr_Current_Axis]
    print(f"running grid search for sensor:{sensor_name}")
    # --- Train/test split ---
    X_train, X_test, y_train, y_test = train_test_split(
        X_arr, Y_arr, test_size=test_size, random_state=random_state, shuffle=True
    )
    best_models_Y_AXIS[sensor_name] = {} 
    for name, (model, params) in models.items():
            results = run_grid_search_per_model(X_train, y_train,cv_number,verbose,name,model,params,PCA__n_components)
            best_models_Y_AXIS[sensor_name][results["name"]] = results

In [ ]:
best_models_Y_AXIS

In [ ]:
import itertools

In [ ]:
def find_best_stacked_regression(models,best_models):
    
    best_result = {
        "score" : -10,
        "model" : '',
        "parameters" : {} 
    }
    
    for final_estimator_name, (final_estimator_model, final_estimator_params) in models.items():
        sensor_models_lists = [
            list(models_for_sensor.items())
            for models_for_sensor in best_models.values()
        ]
        for base_estimators_combination in itertools.product(*sensor_models_lists):
            
            base_estimators = [
                (f"{item[0]}_{i}", item[1]["model"])
                for i, item in enumerate(base_estimators_combination)
            ]
           # print(f"base_estimators{base_estimators}")
            stacked_Estimator = StackingRegressor(
                estimators=base_estimators,
                final_estimator=final_estimator_model,  # we will tune this
                passthrough=True           # include original features if you want
            )
            param_grid = {
                **{'final_estimator' +'__' + k: v for k, v in final_estimator_params.items()}
            }
            print(f"final_estimator_model{stacked_Estimator}")
            print(f"param_grid{param_grid}")
            stacked_grid = GridSearchCV(stacked_Estimator,param_grid,cv=5,verbose=2,n_jobs=-1,scoring='neg_mean_squared_error')
            stacked_grid.fit(X_train, y_train)
        
            
            if stacked_grid.best_score_ > best_result["score"]:
                    print(stacked_grid.best_estimator_)
                    best_result["name"]  = stacked_grid
                    best_result["model"] = stacked_grid.best_estimator_
                    best_result["parameters"] = stacked_grid.best_params_
                    best_result["score"] = stacked_grid.best_score_ 
    return  best_result           

In [ ]:
best_result_X_AXIS ={}

X_arr = np.asarray(X[sensor_name])
Y_arr_full = np.asarray(Y[sensor_name])
Y_arr_Current_Axis = 0
Y_arr = Y_arr_full[:,Y_arr_Current_Axis]
print(f"running grid search for sensor:{sensor_name}")
# --- Train/test split ---
X_train, X_test, y_train, y_test = train_test_split(
    X_arr, Y_arr, test_size=test_size, random_state=random_state, shuffle=True
)
best_result_X_AXIS = find_best_stacked_regression(models,best_models_X_AXIS)

In [ ]:
print(best_result_X_AXIS)

In [ ]:
best_result_Y_AXIS ={}

X_arr = np.asarray(X[sensor_name])
Y_arr_full = np.asarray(Y[sensor_name])
Y_arr_Current_Axis = 1
Y_arr = Y_arr_full[:,Y_arr_Current_Axis]
print(f"running grid search for sensor:{sensor_name}")
# --- Train/test split ---
X_train, X_test, y_train, y_test = train_test_split(
    X_arr, Y_arr, test_size=test_size, random_state=random_state, shuffle=True
)
best_result_Y_AXIS = find_best_stacked_regression(models,best_models_Y_AXIS)

In [ ]:
best_result_Y_AXIS

In [ ]:
X_arr = np.asarray(X[sensor_name])
Y_arr = np.asarray(Y[sensor_name])

X_train, X_test, y_train, y_test = train_test_split(
    X_arr, Y_arr, test_size=test_size, random_state=random_state, shuffle=True
)


PREDICT_POSITION_VALUE = np.empty((Y_arr.shape(0), 2)) 
#predict x axis
PREDICT_POSITION_VALUE[:,0] = TRAINED_MODELS_X_AXIS[sensor_name].predict(best_result_X_AXIS['model'])


#predict y axis

PREDICT_Y_AXIS[sensor_name] = TRAINED_MODELS_Y_AXIS[sensor_name].predict(best_result_Y_AXIS['model'])

PREDICT_POSITION_VALUE[:,1] =  np.column_stack(( PREDICT_X_AXIS[sensor_name], PREDICT_Y_AXIS[sensor_name]))

In [ ]:
plot_pred_vs_actual(POSITION_Y_TEST, PREDICT_POSITION_VALUE,sensor_names,"X-Y AXIS")

In [ ]:
X_AXIS_best_result_per_sensor

In [ ]:
Y_AXIS_best_result_per_sensor

In [ ]:
def find_best_stacked_regression_Pre_Given(models,best_result_per_sensor):
    
    best_result = {
        "score" : -10,
        "model" : '',
        "parameters" : {} 
    }
    
    for final_estimator_name, (final_estimator_model, final_estimator_params) in models.items():
        for base_best_result in best_result_per_sensor.values():
            print(f"b {base_best_result['model']}")
        base_estimators = [(sensor_names,base_best_result['model']) 
                           for sensor_names,base_best_result in best_result_per_sensor.items()]
        print(f"base_estimators:{base_estimators}")
    
       # print(f"base_estimators{base_estimators}")
        stacked_Estimator = StackingRegressor(
            estimators=base_estimators,
            final_estimator=final_estimator_model,  # we will tune this
            passthrough=True           # include original features if you want
        )
        param_grid = {
            **{'final_estimator' +'__' + k: v for k, v in final_estimator_params.items()}
        }
        print(f"final_estimator_model{stacked_Estimator}")
        print(f"param_grid{param_grid}")
        stacked_grid = GridSearchCV(stacked_Estimator,param_grid,cv=5,verbose=2,n_jobs=-1,scoring='neg_mean_squared_error')
        stacked_grid.fit(X_train, y_train)
    
        
        if stacked_grid.best_score_ > best_result["score"]:
                print(stacked_grid.best_estimator_)
                best_result["name"]  = final_estimator_name
                best_result["model"] = stacked_grid.best_estimator_
                best_result["parameters"] = stacked_grid.best_params_
                best_result["score"] = stacked_grid.best_score_ 
    return  best_result           

In [ ]:
best_result_X_AXIS = find_best_stacked_regression_Pre_Given(models,X_AXIS_best_result_per_sensor)

In [ ]:
best_result_X_AXIS

In [ ]:
best_result_Y_AXIS = find_best_stacked_regression_Pre_Given(models,Y_AXIS_best_result_per_sensor)

In [ ]:
best_result_Y_AXIS

In [ ]:
X_arr = np.asarray(X[sensor_name])
Y_arr = np.asarray(Y[sensor_name])

X_train, X_test, y_train, y_test = train_test_split(
    X_arr, Y_arr, test_size=test_size, random_state=random_state, shuffle=True
)


PREDICT_POSITION_VALUE = np.empty((y_test.shape[0], 2)) 
#predict x axis
PREDICT_POSITION_VALUE[:,0] = best_result_X_AXIS['model'].predict(X_test)
PREDICT_POSITION_VALUE[:,0] = PREDICT_POSITION_VALUE[:,0]* (-1)
#predict y axis

PREDICT_POSITION_VALUE[:,1] = best_result_Y_AXIS['model'].predict(X_test)

print(PREDICT_POSITION_VALUE)


In [ ]:
plot_pred_vs_actual(y_test, PREDICT_POSITION_VALUE,sensor_names,"X-Y AXIS")